# MedGemma 4B — Fine-tuning con LoRA / QLoRA (Colab)

Continuación del notebook de inferencia. Aquí entrenamos **adaptadores LoRA** sobre `google/medgemma-4b-it` para experimentar.

**LoRA vs QLoRA:** mismo método (entrenas matrices low-rank y congelas el modelo base). La diferencia es que **QLoRA carga el modelo base en 4-bit (NF4)** → mucha menos VRAM. En Colab free (T4) usa **QLoRA**. Hay un flag `USE_QLORA` para cambiar.

**Importante:** este notebook trae un *dataset de ejemplo mínimo* (placeholder) solo para que el bucle de entrenamiento corra de punta a punta. Sustitúyelo por tu dataset real (REFUGE, etc.) cuando lo tengas — el formato esperado está documentado abajo.

## 1. Instalar dependencias

In [ ]:
!pip install -q -U transformers accelerate timm hf_transfer
!pip install -q -U peft bitsandbytes trl datasets

## 2. Autenticación en Hugging Face (modelo gated)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

from huggingface_hub import login
if token:
    login(token=token)
else:
    login()

## 3. Cargar modelo y processor

Cambia `USE_QLORA` según tu GPU:
- `True`  → QLoRA (base en 4-bit). Recomendado en T4.
- `False` → LoRA puro (base en bf16). Para A100 / GPUs con más VRAM.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-4b-it"
USE_QLORA = True   # <-- cambia a False para LoRA puro (bf16)

processor = AutoProcessor.from_pretrained(MODEL_ID)
# El padding a la izquierda no aplica en entrenamiento (usamos padding normal),
# pero fijamos pad_token por si acaso.
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model_kwargs = dict(
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",  # recomendado para Gemma 3
)

if USE_QLORA:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False  # necesario para gradient checkpointing en entrenamiento
print("Modo:", "QLoRA (4-bit)" if USE_QLORA else "LoRA (bf16)")

## 4. Configurar LoRA y envolver el modelo

- `r` = rango de los adaptadores (capacidad). 8-16 es habitual para empezar.
- `target_modules` = capas donde se inyectan los adaptadores. Aquí: proyecciones de atención + MLP.

Nota: estos nombres (`q_proj`, `k_proj`, ...) aparecen tanto en el LLM como en el encoder de visión (SigLIP), así que se añadirán pequeños adaptadores también a la atención de visión. Para entrenar **solo el lenguaje** (vision tower 100% congelado), usa el `target_modules` con regex que dejo comentado.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    # Alternativa: solo el LLM (congela visión por completo):
    # target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Dataset

**Formato de cada ejemplo:** una imagen + una pregunta (entrada) + una respuesta de referencia (lo que queremos que aprenda a generar).

Construimos un `datasets.Dataset` con 3 columnas simples: `image` (ruta), `question`, `answer`. Las convertimos a mensajes de chat dentro del *collator* (paso 6). Esto mantiene el dataset simple y robusto.

⚠️ Lo de abajo es **placeholder** (reutiliza tu `glaucoma.jpg` con respuestas inventadas) solo para probar el pipeline. Reemplázalo por tu dataset real.

In [ ]:
from datasets import Dataset, Image as HFImage

# Cada registro: ruta a la imagen + pregunta + respuesta objetivo.
# >>> SUSTITUIR por tu dataset real (p.ej. recorrer REFUGE y generar Q/A). <<<
records = [
    {
        "image": "glaucoma.jpg",
        "question": "Describe this fundus image and state whether it suggests glaucoma.",
        "answer": ("This fundus image shows an enlarged optic cup with an increased "
                   "cup-to-disc ratio and thinning of the neuroretinal rim, findings "
                   "consistent with glaucoma."),
    },
    {
        "image": "glaucoma.jpg",
        "question": "What is the cup-to-disc ratio impression here?",
        "answer": "The cup-to-disc ratio appears increased, which is a sign suggestive of glaucoma.",
    },
]

def build_dataset(records):
    ds = Dataset.from_dict({
        "image":    [r["image"] for r in records],
        "question": [r["question"] for r in records],
        "answer":   [r["answer"] for r in records],
    })
    ds = ds.cast_column("image", HFImage())  # las rutas se decodifican a PIL al acceder
    return ds

train_dataset = build_dataset(records)
print(train_dataset)

## 6. Collator (arma el chat + máscara de labels)

Convierte cada ejemplo en una conversación (`user` con imagen+pregunta, `assistant` con la respuesta), tokeniza junto con la imagen y construye `labels`.

Enmascaramos con `-100` (ignorado por la pérdida): tokens de *padding* y los tokens de imagen. Así el modelo solo recibe señal de entrenamiento sobre el texto real.

In [ ]:
# id del token de imagen de Gemma 3 (placeholder de imagen en la secuencia)
IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(
    processor.tokenizer.special_tokens_map["boi_token"]
)

def collate_fn(examples):
    texts, images = [], []
    for ex in examples:
        img = ex["image"].convert("RGB")
        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": ex["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": ex["answer"]},
            ]},
        ]
        text = processor.apply_chat_template(
            messages, add_generation_prompt=False, tokenize=False
        ).strip()
        texts.append(text)
        images.append([img])

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMAGE_TOKEN_ID] = -100
    labels[labels == 262144] = -100  # <image_soft_token> de Gemma 3
    batch["labels"] = labels
    return batch

# Prueba rápida del collator con 1 ejemplo:
_b = collate_fn([train_dataset[0]])
print({k: tuple(v.shape) for k, v in _b.items() if hasattr(v, "shape")})

## 7. Configurar el entrenamiento y entrenar

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="medgemma-glaucoma-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch_fused",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    # Claves para multimodal: no preparar el dataset por su cuenta y no quitar columnas.
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    label_names=["labels"],
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor,
)

In [ ]:
trainer.train()

## 8. Guardar el adaptador LoRA

Solo se guardan los adaptadores (unos pocos MB), no el modelo completo.

In [ ]:
ADAPTER_DIR = "medgemma-glaucoma-lora/adapter"
trainer.save_model(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print("Adaptador guardado en:", ADAPTER_DIR)

# (Opcional) Descargar a tu equipo desde Colab:
# !zip -r adapter.zip medgemma-glaucoma-lora/adapter
# from google.colab import files; files.download('adapter.zip')

## 9. Probar el modelo fine-tuneado

Generamos con el modelo ya con adaptadores cargados (el `model` actual). Reactivamos `use_cache` para que la generación sea eficiente.

In [ ]:
from PIL import Image

model.config.use_cache = True
model.eval()

@torch.inference_mode()
def ask_image(image, question, system_prompt="You are an expert ophthalmologist.",
              max_new_tokens=300, temperature=0.0):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)
    input_len = inputs["input_ids"].shape[-1]
    do_sample = temperature > 0.0
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=do_sample,
                         temperature=temperature if do_sample else None)
    return processor.decode(gen[0][input_len:], skip_special_tokens=True).strip()

image = Image.open("glaucoma.jpg").convert("RGB")
print(ask_image(image, "Describe this fundus image and state whether it suggests glaucoma."))

## 10. (Para sesiones nuevas) Recargar el adaptador sin reentrenar

Si reinicias el runtime, vuelve a cargar el base + el adaptador así:

In [ ]:
# from peft import PeftModel
# base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)
# model = PeftModel.from_pretrained(base, ADAPTER_DIR)
# model.eval()
#
# Para fundir los adaptadores en el base (despliegue, sin dependencia de peft):
# merged = model.merge_and_unload()   # ojo: no funciona si el base está en 4-bit